In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip -q install phonemizer librosa soundfile numba scipy onnx onnxruntime matplotlib tqdm pandas

In [ ]:
import os, math, time, json, random, glob, io, copy, warnings
from dataclasses import dataclass
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import soundfile as sf, librosa
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

@dataclass
class Cfg:
    # audio
    sr: int = 22050; n_fft: int = 1024; hop: int = 256; n_mels: int = 80; fmin: int = 0; fmax: int = 8000
    max_sec: float = 10.0
    # model
    d_model: int = 192; n_heads: int = 4; ffn: int = 768; n_text_layers: int = 4; n_pred_layers: int = 2
    style_dim: int = 128                    # per style encoder (acoustic + prosodic -> 2*style_dim)
    dec_ch: int = 256; up_rates: tuple = (8, 8); up_ks: tuple = (16, 16)
    istft_n_fft: int = 16; istft_hop: int = 4   # 8*8*4 = 256 = hop
    diff_hid: int = 256; diff_layers: int = 4
    # training
    batch: int = 16; lr: float = 2e-4; steps: int = 15000; seg_frames: int = 48
    gan_start: int = 3000; diff_start: int = 1500; warmup: int = 500
    log_every: int = 100; save_every: int = 2500

cfg = Cfg()
assert int(np.prod(cfg.up_rates)) * cfg.istft_hop == cfg.hop

EMOTIONS = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
E2I = {e: i for i, e in enumerate(EMOTIONS)}

# ---- experiment switches -------------------------------------------------------------
STAGE = 2             # 1 = pre-train on LJSpeech (neutral) | 2 = fine-tune on ESD (+ LJSpeech replay) to learn emotions
STAGE_CFG = {1: dict(datasets=['ljspeech'],        steps=40000, lr=2e-4, warmup=500, gan_start=5000, diff_start=3000, w_emo=0.0),
             2: dict(datasets=['esd', 'ljspeech'], steps=15000, lr=1e-4, warmup=200, gan_start=0,    diff_start=0,    w_emo=0.1)}
DATASETS = STAGE_CFG[STAGE]['datasets']      # 'ravdess' also works (add it to stage 2 for calm/fearful/disgust)
for k in ('steps', 'lr', 'warmup', 'gan_start', 'diff_start'): setattr(cfg, k, STAGE_CFG[STAGE][k])
LJ_MAX = 13100        # stage 1: number of LJSpeech utterances (13100 = all)
LJ_REPLAY = 3000      # stage 2: LJSpeech utterances mixed in so general text -> speech is not forgotten
REPLAY_FRAC = 0.3     # stage 2: share of every batch drawn from LJSpeech (the rest is emotional data)
ESD_ROOT = None       # None = download from Kaggle; or a folder that contains 0011/, 0012/, ...
ESD_SPEAKERS = None   # e.g. ['0011','0012','0013','0014'] for a faster subset; None = all 10 English speakers
RESUME = True        # True: continue the CURRENT stage from its last checkpoint (e.g. after a Colab disconnect)
USE_DRIVE = True      # keep checkpoints on Google Drive so they survive disconnects
WORK = '/content'; CACHE = f'{WORK}/cache'; CKPT = f'{WORK}/ckpt'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); CKPT = '/content/drive/MyDrive/styletts2lite_ckpt'
    except Exception as e: print('Drive not mounted, using local disk:', repr(e)[:60])
os.makedirs(CACHE, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
# fixed speaker table -> identical speaker ids in both stages
SPK_LIST = ['lj'] + [f'esd_{i:04d}' for i in range(11, 21)] + [f'rav_{i:02d}' for i in range(1, 25)]
spk2id = {s_: i for i, s_ in enumerate(SPK_LIST)}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print('device:', dev, '| torch', torch.__version__)

In [ ]:
def get_ravdess():
    root = f'{WORK}/ravdess'
    if not glob.glob(f'{root}/**/*.wav', recursive=True):
        os.makedirs(root, exist_ok=True)
        os.system(f'wget -q -O {root}/a.zip "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1" && unzip -q -o {root}/a.zip -d {root}')
    code2emo = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad', '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}
    stmt = {'01': 'Kids are talking by the door.', '02': 'Dogs are sitting by the door.'}
    items = []
    for p in sorted(glob.glob(f'{root}/**/*.wav', recursive=True)):
        f = os.path.basename(p)[:-4].split('-')
        if len(f) == 7:
            items.append(dict(wav=p, text=stmt[f[4]], emo=code2emo[f[2]], spk='rav_' + f[6]))
    return items

def get_ljspeech(max_utts=None):
    root = f'{WORK}/LJSpeech-1.1'
    if not os.path.exists(root):
        os.system(f'wget -q -O {WORK}/lj.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2 && tar -xjf {WORK}/lj.tar.bz2 -C {WORK}')
    items = []
    for l in open(f'{root}/metadata.csv', encoding='utf8'):
        i, raw, norm = l.rstrip('\n').split('|')
        items.append(dict(wav=f'{root}/wavs/{i}.wav', text=norm, emo='neutral', spk='lj'))
    return items[:max_utts] if max_utts else items

def download_esd():
    """Kaggle mirror of ESD (3.4 GB). Needs Kaggle credentials: either upload kaggle.json to ~/.kaggle/ (chmod 600)
    or set KAGGLE_USERNAME / KAGGLE_KEY env vars (Colab: kaggle.com -> Settings -> Create New Token)."""
    root = f'{WORK}/esd_raw'
    if not glob.glob(f'{root}/**/*.wav', recursive=True):
        os.makedirs(root, exist_ok=True)
        os.system(f'pip -q install kaggle && kaggle datasets download -d nguyenthanhlim/emotional-speech-dataset-esd -p {root} --unzip')
    return root

def get_esd(root):
    emap = {'neutral': 'neutral', 'happy': 'happy', 'angry': 'angry', 'sad': 'sad', 'surprise': 'surprised'}
    spk_dirs = sorted(d for d in glob.glob(f'{root}/**/00[0-9][0-9]', recursive=True) if os.path.isdir(d))
    spk_dirs = [d for d in spk_dirs if 11 <= int(os.path.basename(d)) <= 20]          # 0001-0010 are Mandarin
    if ESD_SPEAKERS: spk_dirs = [d for d in spk_dirs if os.path.basename(d) in ESD_SPEAKERS]
    items = []
    for sd in spk_dirs:
        spk = os.path.basename(sd); folders = {n.lower(): n for n in os.listdir(sd) if os.path.isdir(f'{sd}/{n}')}
        txts = glob.glob(f'{sd}/*.txt')
        if not txts: print('no transcript in', sd); continue
        for line in open(txts[0], encoding='utf8', errors='ignore'):
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3 or parts[2].strip().lower() not in emap: continue
            uid, text, emo = parts[0].strip(), parts[1].strip(), parts[2].strip().lower()
            p = f'{sd}/{folders.get(emo, emo)}/{uid}.wav'
            if os.path.exists(p): items.append(dict(wav=p, text=text, emo=emap[emo], spk='esd_' + spk))
    print(f'ESD: {len(items)} utterances from {len(spk_dirs)} speakers')
    if not items: print('!! ESD parsed 0 utterances. Run: !find', root, '-maxdepth 3 | head -30   and check the folder layout')
    return items

def build_items():
    items = []
    if 'ravdess' in DATASETS: items += get_ravdess()
    if 'ljspeech' in DATASETS: items += get_ljspeech(LJ_MAX if STAGE == 1 else LJ_REPLAY)
    if 'esd' in DATASETS: items += get_esd(ESD_ROOT or download_esd())
    return items

In [ ]:
try:
    from phonemizer.backend import EspeakBackend
    _bk = EspeakBackend('en-us', preserve_punctuation=True, with_stress=True)
    def phonemize(texts): return _bk.phonemize(list(texts), strip=True)
    PHON = True
except Exception as e:
    print('espeak/phonemizer not available -> using graphemes:', repr(e)[:80]); PHON = False
    def phonemize(texts): return [t.lower() for t in texts]

_mel_basis = librosa.filters.mel(sr=cfg.sr, n_fft=cfg.n_fft, n_mels=cfg.n_mels, fmin=cfg.fmin, fmax=cfg.fmax)

class MelSpec(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('basis', torch.tensor(_mel_basis, dtype=torch.float32))
        self.register_buffer('win', torch.hann_window(cfg.n_fft))
    def forward(self, wav):                                   # [B,T] -> [B,80,frames] log-mel
        X = torch.stft(wav.float(), cfg.n_fft, cfg.hop, cfg.n_fft, self.win, center=True, return_complex=True)
        mag = torch.sqrt(X.real ** 2 + X.imag ** 2 + 1e-9)
        return torch.log(torch.clamp(self.basis @ mag, min=1e-5))

def norm_mel(m): return (m + 4.0) / 4.0                       # roughly zero-mean / unit-std

_MEL_CPU = MelSpec()

def extract(item):
    y, _ = librosa.load(item['wav'], sr=cfg.sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=35)
    if len(y) < 0.4 * cfg.sr or len(y) > cfg.max_sec * cfg.sr: return None
    y = (y / (np.abs(y).max() + 1e-8) * 0.95).astype(np.float32)
    mel = _MEL_CPU(torch.from_numpy(y)[None])[0]              # [80,Fm]
    Fm = mel.shape[1]
    def fit(a, fill):
        a = a[:Fm]; return np.pad(a, (0, Fm - len(a)), constant_values=fill) if len(a) < Fm else a
    f0 = fit(librosa.yin(y, fmin=60, fmax=500, sr=cfg.sr, frame_length=1024, hop_length=cfg.hop), 0)
    rms = fit(librosa.feature.rms(y=y, frame_length=1024, hop_length=cfg.hop)[0], 0)
    voiced = rms > 0.04 * rms.max()
    f0n = np.where(voiced, 3 * np.log10(np.clip(f0, 50, 600) / 50), 0)     # 0 = unvoiced, voiced in ~[0.24, 3.2]
    from scipy.signal import medfilt
    f0n = medfilt(f0n, 5)
    en = np.log10(rms + 1e-4) + 4.0
    return dict(wav=torch.from_numpy((y * 32767).astype(np.int16)), mel=mel.half(), f0=torch.from_numpy(f0n).float(),
                en=torch.from_numpy(en).float())

import hashlib
def preprocess(items):
    """Features are cached per wav path, so LJSpeech is only processed once and reused by stage 2.
    (If you change any audio setting in Cfg, delete /content/cache.)"""
    metas = []
    for it in tqdm(items, desc='features'):
        path = f"{CACHE}/{hashlib.md5(it['wav'].encode()).hexdigest()[:16]}.pt"
        if not os.path.exists(path):
            try: d = extract(it)
            except Exception as e: print('skip', it['wav'], repr(e)[:60]); continue
            if d is None: continue
            torch.save(d, path)
        assert it['spk'] in spk2id, f"unknown speaker {it['spk']}: add it to SPK_LIST"
        metas.append(dict(path=path, text=it['text'], emo_name=it['emo'], spk_name=it['spk'], emo=E2I[it['emo']], spk=spk2id[it['spk']]))
    uniq = sorted({m['text'] for m in metas}); ph = dict(zip(uniq, phonemize(uniq)))       # phonemize each unique text once
    for m in metas: m['phon'] = ph[m['text']]
    return metas

In [ ]:
class TTSData(torch.utils.data.Dataset):
    def __init__(self, metas, c2i): self.m, self.c2i = metas, c2i
    def __len__(self): return len(self.m)
    def __getitem__(self, i):
        m = self.m[i]; d = torch.load(m['path'])
        ids = torch.tensor([self.c2i.get(c, 1) for c in m['phon']])
        return dict(ids=ids, wav=d['wav'].float() / 32767, mel=d['mel'].float(), f0=d['f0'], en=d['en'], emo=m['emo'], spk=m['spk'])

def collate(batch):
    B = len(batch); Tt = max(len(b['ids']) for b in batch); Tm = max(b['mel'].shape[1] for b in batch)
    out = dict(ids=torch.zeros(B, Tt, dtype=torch.long), mel=torch.full((B, cfg.n_mels, Tm), math.log(1e-5)),
               f0=torch.zeros(B, Tm), en=torch.zeros(B, Tm), wav=torch.zeros(B, Tm * cfg.hop),
               tlen=torch.zeros(B, dtype=torch.long), mlen=torch.zeros(B, dtype=torch.long),
               emo=torch.tensor([b['emo'] for b in batch]), spk=torch.tensor([b['spk'] for b in batch]))
    for i, b in enumerate(batch):
        t, m = len(b['ids']), b['mel'].shape[1]
        out['ids'][i, :t] = b['ids']; out['mel'][i, :, :m] = b['mel']; out['f0'][i, :m] = b['f0']; out['en'][i, :m] = b['en']
        out['wav'][i, :len(b['wav'])] = b['wav']; out['tlen'][i] = t; out['mlen'][i] = m
    return out

In [ ]:
def sinusoid(T, d, device):
    pos = torch.arange(T, device=device).float()[:, None]; i = torch.arange(0, d, 2, device=device).float()[None]
    ang = pos / (10000 ** (i / d)); pe = torch.zeros(T, d, device=device)
    pe[:, 0::2] = torch.sin(ang); pe[:, 1::2] = torch.cos(ang); return pe

class AdaLN(nn.Module):
    def __init__(s, d, sd):
        super().__init__(); s.ln = nn.LayerNorm(d, elementwise_affine=False); s.fc = nn.Linear(sd, 2 * d)
        nn.init.zeros_(s.fc.weight); nn.init.zeros_(s.fc.bias)
    def forward(s, x, sty):
        g, b = s.fc(sty).unsqueeze(1).chunk(2, -1); return (1 + g) * s.ln(x) + b

class Block(nn.Module):
    """Pre-LN transformer block; masks padding; AdaLN if sd>0 (style conditioning)."""
    def __init__(s, d, h, ffn, sd=0, p=0.1):
        super().__init__(); s.sd, s.h = sd, h
        s.n1 = AdaLN(d, sd) if sd else nn.LayerNorm(d); s.n2 = AdaLN(d, sd) if sd else nn.LayerNorm(d)
        s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d)
        s.c1 = nn.Conv1d(d, ffn, 3, padding=1); s.c2 = nn.Conv1d(ffn, d, 1); s.dp = nn.Dropout(p)
    def _n(s, n, x, sty): return n(x, sty) if s.sd else n(x)
    def forward(s, x, m, sty=None):                            # x [B,T,D], m [B,T] bool (True = valid)
        B, T, D = x.shape
        q, k, v = s.qkv(s._n(s.n1, x, sty)).view(B, T, 3, s.h, D // s.h).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=m[:, None, None, :])
        x = x + s.dp(s.o(y.transpose(1, 2).reshape(B, T, D)))
        mm = m.unsqueeze(-1).to(x.dtype)
        h = (s._n(s.n2, x, sty) * mm).transpose(1, 2)
        x = x + s.dp(s.c2(F.relu(s.c1(h))).transpose(1, 2))
        return x * mm

class TextEncoder(nn.Module):
    def __init__(s, V, d, h, ffn, L):
        super().__init__(); s.d = d
        s.emb = nn.Embedding(V, d, padding_idx=0); nn.init.normal_(s.emb.weight, 0, d ** -0.5)
        s.blocks = nn.ModuleList([Block(d, h, ffn) for _ in range(L)]); s.ln = nn.LayerNorm(d)
    def forward(s, ids, m):
        e = s.emb(ids); x = e * math.sqrt(s.d) + sinusoid(ids.shape[1], s.d, ids.device)
        for b in s.blocks: x = b(x, m)
        return s.ln(x) * m.unsqueeze(-1), e

class StyleEncoder(nn.Module):
    """mel -> style vector. Dilated convs + masked mean/std pooling (no temporal info leaks through)."""
    def __init__(s, n_mels, dim, ch=128):
        super().__init__()
        s.c0 = nn.Conv1d(n_mels, ch, 5, padding=2)
        s.cs = nn.ModuleList([nn.Conv1d(ch, ch, 5, padding=2 * dl, dilation=dl) for dl in (1, 2, 4)])
        s.fc = nn.Linear(2 * ch, dim)
    def forward(s, mel, m):                                    # mel [B,80,T] normalised, m [B,T]
        x = F.leaky_relu(s.c0(mel), 0.2)
        for c in s.cs: x = x + F.leaky_relu(c(x), 0.2)
        mm = m.unsqueeze(1).to(x.dtype); n = mm.sum(-1).clamp(min=1)
        mu = (x * mm).sum(-1) / n; sd = (((x - mu.unsqueeze(-1)) ** 2 * mm).sum(-1) / n + 1e-5).sqrt()
        return s.fc(torch.cat([mu, sd], -1))

class Aligner(nn.Module):
    """FastPitch/RAD-TTS style learned text-mel aligner (replaces StyleTTS2's ASR-pretrained aligner)."""
    def __init__(s, d, n_mels=80, c=80):
        super().__init__()
        s.key = nn.Sequential(nn.Conv1d(d, 2 * d, 3, padding=1), nn.ReLU(), nn.Conv1d(2 * d, c, 1))
        s.qry = nn.Sequential(nn.Conv1d(n_mels, 2 * c, 3, padding=1), nn.ReLU(), nn.Conv1d(2 * c, c, 1), nn.ReLU(), nn.Conv1d(c, c, 1))
        s.log_t = nn.Parameter(torch.tensor(math.log(0.05)))
    def forward(s, emb, mel):
        k = s.key(emb.transpose(1, 2)); q = s.qry(mel)
        d2 = (q ** 2).sum(1)[:, :, None] + (k ** 2).sum(1)[:, None, :] - 2 * torch.einsum('bcm,bct->bmt', q, k)
        return -s.log_t.exp() * d2.clamp(min=0)               # [B,Tm,Tt]

class DurationPredictor(nn.Module):
    def __init__(s, d, h, ffn, sd, L):
        super().__init__(); s.blocks = nn.ModuleList([Block(d, h, ffn, sd) for _ in range(L)]); s.out = nn.Linear(d, 1)
    def forward(s, x, m, sty):
        for b in s.blocks: x = b(x, m, sty)
        return s.out(x).squeeze(-1)                            # log(1+dur)

class ProsodyPredictor(nn.Module):
    def __init__(s, d, h, ffn, sd, L):
        super().__init__(); s.blocks = nn.ModuleList([Block(d, h, ffn, sd) for _ in range(L)])
        s.f0 = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        s.en = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
    def forward(s, x, m, sty):                                 # x: frame-level (aligned) text features [B,Tm,D]
        x = (x + sinusoid(x.shape[1], x.shape[2], x.device)) * m.unsqueeze(-1)
        for b in s.blocks: x = b(x, m, sty)
        return s.f0(x).squeeze(-1), s.en(x).squeeze(-1)

In [ ]:
import numba

@numba.jit(nopython=True)
def _mas(path, value, t_y, t_x):                                # Glow-TTS / VITS style MAS; value [t_x, t_y]
    NEG = -1e9; index = t_x - 1
    for y in range(t_y):
        for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
            v_cur = NEG if x == y else value[x, y - 1]
            if x == 0: v_prev = 0. if y == 0 else NEG
            else: v_prev = value[x - 1, y - 1]
            value[x, y] = max(v_cur, v_prev) + value[x, y]
    for y in range(t_y - 1, -1, -1):
        path[index, y] = 1
        if index != 0 and (index == y or value[index, y - 1] < value[index - 1, y - 1]): index -= 1

def mas_batch(logp, tl, ml):                                    # logp [B,Tm,Tt] -> hard alignment [B,Tm,Tt]
    B, Tm, Tt = logp.shape; lp = logp.detach().float().cpu().numpy(); out = np.zeros((B, Tm, Tt), np.float32)
    for i in range(B):
        ty, tx = int(ml[i]), int(tl[i])
        if ty < tx:                                             # degenerate; uniform fallback
            out[i, np.arange(ty), np.floor(np.arange(ty) * tx / ty).astype(int)] = 1; continue
        v = np.ascontiguousarray(lp[i, :ty, :tx].T); p = np.zeros((tx, ty), np.float32)
        _mas(p, v, ty, tx); out[i, :ty, :tx] = p.T
    return torch.from_numpy(out).to(logp.device)

def log_beta_binomial(tl, ml, Tt, Tm, scaling=1.0):             # diagonal prior (FastPitch); returns [B,Tm,Tt] log-prob
    dev = tl.device; lg = torch.lgamma
    i = torch.arange(1, Tm + 1, device=dev).float()[None, :, None]; k = torch.arange(0, Tt, device=dev).float()[None, None, :]
    n = tl.float()[:, None, None]; M = ml.float()[:, None, None]
    a = scaling * i; b = (scaling * (M + 1 - i)).clamp(min=1e-3); kk = torch.minimum(k, n)
    lbeta = lambda x, y: lg(x) + lg(y) - lg(x + y)
    return lg(n + 1) - lg(kk + 1) - lg(n - kk + 1) + lbeta(kk + a, n - kk + b) - lbeta(a, b)

def forward_sum_loss(logp, tl, ml, blank=-1.0):                 # logp [B,Tm,Tt] (log-probs over text axis)
    B, Tm, Tt = logp.shape
    lp = F.pad(logp, (1, 0), value=blank)                       # blank at class 0
    cls = torch.arange(Tt + 1, device=lp.device)[None, None, :]
    lp = lp.masked_fill(cls > tl[:, None, None], -1e4)
    lp = F.log_softmax(lp, -1).permute(1, 0, 2).float()         # [Tm,B,Tt+1]
    tgt = torch.arange(1, Tt + 1, device=lp.device)[None].expand(B, -1)
    return F.ctc_loss(lp, tgt, ml, tl, zero_infinity=True)

In [ ]:
class AdaIN(nn.Module):
    def __init__(s, sd, ch):
        super().__init__(); s.norm = nn.InstanceNorm1d(ch, affine=False); s.fc = nn.Linear(sd, 2 * ch)
    def forward(s, x, sty):
        g, b = s.fc(sty).unsqueeze(-1).chunk(2, 1); return (1 + g) * s.norm(x) + b

class Snake(nn.Module):
    def __init__(s, ch): super().__init__(); s.a = nn.Parameter(torch.ones(1, ch, 1))
    def forward(s, x): return x + (1.0 / (s.a + 1e-6)) * torch.sin(s.a * x) ** 2

class AdaResBlk(nn.Module):
    def __init__(s, cin, cout, sd):
        super().__init__(); s.a1 = AdaIN(sd, cin); s.c1 = nn.Conv1d(cin, cout, 3, padding=1)
        s.a2 = AdaIN(sd, cout); s.c2 = nn.Conv1d(cout, cout, 3, padding=1)
        s.sk = nn.Conv1d(cin, cout, 1, bias=False) if cin != cout else nn.Identity()
    def forward(s, x, sty):
        h = s.c1(F.leaky_relu(s.a1(x, sty), 0.2)); h = s.c2(F.leaky_relu(s.a2(h, sty), 0.2))
        return (s.sk(x) + h) / math.sqrt(2)

class GenRes(nn.Module):
    def __init__(s, ch, k, dils, sd):
        super().__init__()
        s.a1 = nn.ModuleList([AdaIN(sd, ch) for _ in dils]); s.a2 = nn.ModuleList([AdaIN(sd, ch) for _ in dils])
        s.c1 = nn.ModuleList([nn.Conv1d(ch, ch, k, padding=d * (k - 1) // 2, dilation=d) for d in dils])
        s.c2 = nn.ModuleList([nn.Conv1d(ch, ch, k, padding=(k - 1) // 2) for _ in dils])
        s.s1 = nn.ModuleList([Snake(ch) for _ in dils]); s.s2 = nn.ModuleList([Snake(ch) for _ in dils])
    def forward(s, x, sty):
        for a1, c1, a2, c2, s1, s2 in zip(s.a1, s.c1, s.a2, s.c2, s.s1, s.s2):
            h = c1(s1(a1(x, sty))); h = c2(s2(a2(h, sty))); x = x + h
        return x

class Decoder(nn.Module):
    """frame-level text features + F0 + energy + acoustic style -> waveform (iSTFTNet: predicts magnitude & phase)."""
    def __init__(s, c):
        super().__init__(); s.n_fft, s.hop = c.istft_n_fft, c.istft_hop; sd = c.style_dim; ch = c.dec_ch
        s.f0c = nn.Conv1d(1, 32, 3, padding=1); s.enc = nn.Conv1d(1, 32, 3, padding=1)
        s.pre = nn.Conv1d(c.d_model + 64, ch, 3, padding=1); s.b1 = AdaResBlk(ch, ch, sd); s.b2 = AdaResBlk(ch, ch, sd)
        s.ups = nn.ModuleList(); s.res = nn.ModuleList(); cc = ch
        for u, k in zip(c.up_rates, c.up_ks):
            s.ups.append(nn.ConvTranspose1d(cc, cc // 2, k, u, padding=(k - u) // 2)); cc //= 2
            s.res.append(nn.ModuleList([GenRes(cc, kk, dd, sd) for kk, dd in [(3, (1, 3)), (7, (1, 3))]]))
        s.sn = Snake(cc); s.post = nn.Conv1d(cc, 2 * (s.n_fft // 2 + 1), 7, padding=3)
        s.register_buffer('win', torch.hann_window(s.n_fft)); s.override = None
    def core(s, asr, f0, en, sty):                              # asr [B,D,T] f0,en [B,1,T] sty [B,S] -> raw [B,2*(n_fft/2+1), T*64]
        x = s.pre(torch.cat([asr, s.f0c(f0), s.enc(en)], 1)); x = s.b2(s.b1(x, sty), sty)
        for up, rbs in zip(s.ups, s.res):
            x = up(F.leaky_relu(x, 0.1)); x = sum(rb(x, sty) for rb in rbs) / len(rbs)
        return s.post(F.leaky_relu(s.sn(x), 0.01))
    def to_wav(s, raw, length=None):
        n = s.n_fft // 2 + 1; mag = torch.exp(raw[:, :n].clamp(max=8.0)); ph = torch.sin(raw[:, n:])
        spec = torch.complex(mag * torch.cos(ph), mag * torch.sin(ph))
        return torch.istft(spec, s.n_fft, s.hop, s.n_fft, window=s.win, center=True, length=length)
    def forward(s, asr, f0, en, sty, length=None):
        raw = s.override(asr, f0, en, sty) if s.override is not None else s.core(asr, f0, en, sty)
        return s.to_wav(raw.float(), length)

from torch.nn.utils.parametrizations import weight_norm as _wn
class PeriodD(nn.Module):
    def __init__(s, p):
        super().__init__(); s.p = p; ch = [1, 32, 64, 128, 256]
        s.convs = nn.ModuleList([_wn(nn.Conv2d(ch[i], ch[i + 1], (5, 1), (3, 1), padding=(2, 0))) for i in range(4)] + [_wn(nn.Conv2d(256, 256, (5, 1), 1, padding=(2, 0)))])
        s.post = _wn(nn.Conv2d(256, 1, (3, 1), padding=(1, 0)))
    def forward(s, x):
        B, C, T = x.shape
        if T % s.p: x = F.pad(x, (0, s.p - T % s.p), 'reflect')
        x = x.view(B, C, -1, s.p); fm = []
        for c in s.convs: x = F.leaky_relu(c(x), 0.1); fm.append(x)
        x = s.post(x); fm.append(x); return x.flatten(1), fm

class MPD(nn.Module):
    def __init__(s, periods=(2, 3, 5, 7, 11)): super().__init__(); s.ds = nn.ModuleList([PeriodD(p) for p in periods])
    def forward(s, y):
        o, f = zip(*[d(y.unsqueeze(1)) for d in s.ds]); return list(o), list(f)

In [ ]:
class DBlock(nn.Module):
    def __init__(s, h):
        super().__init__(); s.ln = nn.LayerNorm(h, elementwise_affine=False); s.mod = nn.Linear(h, 3 * h)
        s.ff = nn.Sequential(nn.Linear(h, 2 * h), nn.SiLU(), nn.Linear(2 * h, h)); nn.init.zeros_(s.mod.weight); nn.init.zeros_(s.mod.bias)
    def forward(s, x, c):
        sh, sc, g = s.mod(c).chunk(3, -1); return x + g * s.ff(s.ln(x) * (1 + sc) + sh)

class StyleDiffusion(nn.Module):
    def __init__(s, D, d_text, n_emo, n_spk, h=256, L=4):
        super().__init__(); s.D, s.n_emo = D, n_emo
        s.emo = nn.Embedding(n_emo + 1, 64); s.spk = nn.Embedding(n_spk, 64)     # index n_emo = null emotion
        s.cond = nn.Sequential(nn.Linear(d_text + 128, h), nn.SiLU(), nn.Linear(h, h))
        s.tmlp = nn.Sequential(nn.Linear(16, h), nn.SiLU(), nn.Linear(h, h))
        s.inp = nn.Linear(D, h); s.blocks = nn.ModuleList([DBlock(h) for _ in range(L)]); s.out = nn.Linear(h, D)
        s.register_buffer('mu', torch.zeros(D)); s.register_buffer('sd', torch.ones(D)); s.register_buffer('init', torch.tensor(0))
        s.register_buffer('freqs', 2.0 ** torch.arange(8).float())
    @torch.no_grad()
    def update_stats(s, x, m=0.99):
        sd = (x - x.mean(0)).std().clamp(min=1e-2).expand_as(s.sd)        # one global scale -> robust when dims are still collapsed
        if s.init.item() == 0: s.mu.copy_(x.mean(0)); s.sd.copy_(sd); s.init.fill_(1)
        else: s.mu.mul_(m).add_((1 - m) * x.mean(0)); s.sd.mul_(m).add_((1 - m) * sd)
    def net(s, x, cn, tp, e, spk):
        f = cn[:, None] * s.freqs[None]; t = s.tmlp(torch.cat([torch.sin(f), torch.cos(f)], -1))
        c = s.cond(torch.cat([tp, s.emo(e), s.spk(spk)], -1)) + t
        h = s.inp(x)
        for b in s.blocks: h = b(h, c)
        return s.out(h)
    def denoise(s, x, sig, tp, e, spk):                          # sigma_data = 1 (styles are standardised)
        c_skip = 1 / (sig ** 2 + 1); c_out = sig / torch.sqrt(sig ** 2 + 1); c_in = 1 / torch.sqrt(sig ** 2 + 1)
        return c_skip * x + c_out * s.net(c_in * x, torch.log(sig.squeeze(-1)) / 4, tp, e, spk)
    def loss(s, style, tp, emo, spk, p_drop=0.1):
        x0 = (style - s.mu) / s.sd; B = x0.shape[0]
        sig = torch.exp(torch.randn(B, 1, device=x0.device) * 1.2 - 1.2)
        e = torch.where(torch.rand(B, device=x0.device) < p_drop, torch.full_like(emo, s.n_emo), emo)
        D = s.denoise(x0 + sig * torch.randn_like(x0), sig, tp, e, spk)
        return (((sig ** 2 + 1) / sig ** 2) * (D - x0) ** 2).mean()
    @torch.no_grad()
    def sample(s, tp, emo, spk, steps=8, cfg_scale=1.5, smin=0.002, smax=8.0, rho=7.0):
        B, dev = tp.shape[0], tp.device
        t = torch.linspace(0, 1, steps, device=dev); sig = (smax ** (1 / rho) + t * (smin ** (1 / rho) - smax ** (1 / rho))) ** rho
        sig = torch.cat([sig, torch.zeros(1, device=dev)]); x = torch.randn(B, s.D, device=dev) * sig[0]
        null = torch.full_like(emo, s.n_emo)
        def Dn(x, sg):
            sg = sg.reshape(1, 1).expand(B, 1); c = s.denoise(x, sg, tp, emo, spk)
            if cfg_scale == 1.0: return c
            u = s.denoise(x, sg, tp, null, spk); return u + cfg_scale * (c - u)
        for i in range(steps):                                   # Heun (2nd order) sampler
            sg, sn = sig[i], sig[i + 1]; d = (x - Dn(x, sg)) / sg; xn = x + (sn - sg) * d
            if sn > 0: xn = x + (sn - sg) * (d + (xn - Dn(xn, sn)) / sn) / 2
            x = xn
        return x * s.sd + s.mu

In [ ]:
def stft_loss(x, y, n, h):
    w = torch.hann_window(n, device=x.device); X = torch.stft(x, n, h, n, w, return_complex=True); Y = torch.stft(y, n, h, n, w, return_complex=True)
    Xm = torch.sqrt(X.real ** 2 + X.imag ** 2 + 1e-7); Ym = torch.sqrt(Y.real ** 2 + Y.imag ** 2 + 1e-7)
    return torch.norm(Ym - Xm) / (torch.norm(Ym) + 1e-7) + F.l1_loss(torch.log(Xm), torch.log(Ym))

def mr_stft(x, y): return sum(stft_loss(x, y, n, h) for n, h in [(512, 128), (1024, 256), (2048, 512)]) / 3

def masked_l1(a, b, m): return ((a - b).abs() * m).sum() / m.sum().clamp(min=1)

class StyleTTS2Lite(nn.Module):
    def __init__(s, c, V, n_emo, n_spk):
        super().__init__(); s.c = c; d, h, sd = c.d_model, c.n_heads, c.style_dim
        s.mel = MelSpec()
        s.text_enc = TextEncoder(V, d, h, c.ffn, c.n_text_layers)
        s.sty_a = StyleEncoder(c.n_mels, sd); s.sty_p = StyleEncoder(c.n_mels, sd)
        s.aligner = Aligner(d)
        s.dur = DurationPredictor(d, h, c.ffn, sd, c.n_pred_layers); s.pros = ProsodyPredictor(d, h, c.ffn, sd, c.n_pred_layers)
        s.decoder = Decoder(c)
        s.diff = StyleDiffusion(2 * sd, d, n_emo, n_spk, c.diff_hid, c.diff_layers)
        s.emo_head = nn.Linear(2 * sd, n_emo)

    def _style(s, mel, m):
        mn = norm_mel(mel) * m.unsqueeze(1); return torch.cat([s.sty_a(mn, m), s.sty_p(mn, m)], -1)

    def _align(s, emb, mn, tm, tl, ml):
        Tt, Tm = emb.shape[1], mn.shape[2]
        logits = s.aligner(emb, mn).masked_fill(~tm[:, None, :], -1e4)
        logp = F.log_softmax(logits, -1) + log_beta_binomial(tl, ml, Tt, Tm)
        logp = logp.masked_fill(~tm[:, None, :], -1e4)
        with torch.no_grad(): hard = mas_batch(F.log_softmax(logp, -1), tl, ml)
        return logp, hard

    def train_forward(s, b, use_diff=False):
        c = s.c; ids, tl, mel, ml = b['ids'], b['tlen'], b['mel'], b['mlen']; B, Tm = ids.shape[0], mel.shape[2]
        tm = torch.arange(ids.shape[1], device=ids.device)[None] < tl[:, None]
        mm = torch.arange(Tm, device=ids.device)[None] < ml[:, None]
        text_h, emb = s.text_enc(ids, tm)
        sty = s._style(mel, mm); sa, sp = sty.split(c.style_dim, -1)
        logp, hard = s._align(emb, norm_mel(mel) * mm.unsqueeze(1), tm, tl, ml)
        dur = hard.sum(1); asr = torch.bmm(hard, text_h)                       # [B,Tt], [B,Tm,D]
        L = {'fs': forward_sum_loss(logp, tl, ml)}
        L['dur'] = masked_l1(s.dur(text_h, tm, sp), torch.log1p(dur), tm.float())
        f0p, enp = s.pros(asr, mm, sp)
        L['f0'] = masked_l1(f0p, b['f0'], mm.float()); L['en'] = masked_l1(enp, b['en'], mm.float())
        # decoder trained on a random fixed-length segment (as in StyleTTS 2)
        seg = int(min(c.seg_frames, ml.min().item())); ar = torch.arange(B, device=ids.device)[:, None]
        st = (torch.rand(B, device=ids.device) * (ml - seg + 1).float()).long()
        fi = st[:, None] + torch.arange(seg, device=ids.device)[None]
        wi = st[:, None] * c.hop + torch.arange(seg * c.hop, device=ids.device)[None]
        wav_g = s.decoder(asr[ar, fi].transpose(1, 2), b['f0'][ar, fi][:, None], b['en'][ar, fi][:, None], sa, length=seg * c.hop)
        wav_r = b['wav'][ar, wi]
        L['mel'] = F.l1_loss(s.mel(wav_g), s.mel(wav_r)); L['stft'] = mr_stft(wav_g, wav_r)
        L['emo'] = F.cross_entropy(s.emo_head(sty), b['emo'])
        if use_diff:
            s.diff.update_stats(sty.detach())
            tp = (text_h.detach() * tm.unsqueeze(-1)).sum(1) / tm.sum(1, keepdim=True)
            L['diff'] = s.diff.loss(sty.detach(), tp, b['emo'], b['spk'])
        return L, wav_g, wav_r

    @torch.no_grad()
    def infer(s, ids, emo=None, spk=None, ref_mel=None, steps=8, cfg_scale=1.5, speed=1.0):
        c = s.c; tm = torch.ones_like(ids, dtype=torch.bool); text_h, _ = s.text_enc(ids, tm)
        if ref_mel is not None:                                                # style transfer from a reference
            sty = s._style(ref_mel, torch.ones(1, ref_mel.shape[2], dtype=torch.bool, device=ids.device))
        else:                                                                   # reference-free: style diffusion
            sty = s.diff.sample(text_h.mean(1), emo, spk, steps, cfg_scale)
        sa, sp = sty.split(c.style_dim, -1)
        d = (torch.expm1(s.dur(text_h, tm, sp)).clamp(min=0) / speed).round().clamp(min=1).long()
        asr = torch.repeat_interleave(text_h[0], d[0], dim=0)[None]; mm = torch.ones(1, asr.shape[1], dtype=torch.bool, device=ids.device)
        f0, en = s.pros(asr, mm, sp); f0 = torch.where(f0 < 0.15, torch.zeros_like(f0), f0)
        wav = s.decoder(asr.transpose(1, 2), f0[:, None], en[:, None], sa, length=asr.shape[1] * c.hop)
        return wav[0], dict(dur=d, style=sty)

def count_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

In [ ]:
import string
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
src_of = lambda m: m['spk_name'].split('_')[0]                # 'lj' | 'esd' | 'rav'
items = build_items(); print(len(items), 'utterances listed')
metas = preprocess(items)
if STAGE == 2:                                                # vocabulary + weights come from the stage-1 checkpoint
    ck1 = torch.load(f'{CKPT}/stage1_final.pt', map_location='cpu'); vocab = ck1['vocab']
    arch = ['d_model', 'n_heads', 'ffn', 'n_text_layers', 'n_pred_layers', 'style_dim', 'dec_ch', 'diff_hid', 'diff_layers']
    bad = [k for k in arch if ck1['cfg'][k] != getattr(cfg, k)]; assert not bad, f'architecture differs from the stage-1 checkpoint: {bad}'
else:                                                         # reserve all punctuation so stage 2 never hits <unk> for it
    vocab = ['<pad>', '<unk>'] + sorted(set(string.punctuation + ' ') | {ch for m in metas for ch in m['phon']})
c2i = {ch: i for i, ch in enumerate(vocab)}
unk = Counter(ch for m in metas for ch in m['phon'] if ch not in c2i)
print(f'{len(metas)} utterances | vocab {len(vocab)} | phonemizer={PHON} | sources {dict(Counter(src_of(m) for m in metas))}')
if unk: print('symbols missing from the stage-1 vocabulary (mapped to <unk>):', dict(unk))

# hold out whole SENTENCES (per source) so validation/evaluation text is never seen in training, in any emotion or voice
rng = random.Random(0); val_texts = set()
for s_ in {src_of(m) for m in metas}:
    tx = sorted({m['text'] for m in metas if src_of(m) == s_}); rng.shuffle(tx)
    val_texts |= set(tx[:min(max(10, int(0.03 * len(tx))), max(1, len(tx) // 5))])
val_idx = [i for i, m in enumerate(metas) if m['text'] in val_texts]; tr_idx = [i for i, m in enumerate(metas) if m['text'] not in val_texts]
print(f'train {len(tr_idx)} utts | held-out {len(val_idx)} utts ({len(val_texts)} unseen sentences)')

NW = 2 if dev == 'cuda' else 0
mk = lambda ii, sh: DataLoader(TTSData([metas[i] for i in ii], c2i), batch_size=cfg.batch, shuffle=sh, collate_fn=collate, num_workers=NW, drop_last=sh)
def make_train_loader():
    ds = TTSData([metas[i] for i in tr_idx], c2i); srcs = [src_of(metas[i]) for i in tr_idx]
    n_lj = sum(s_ == 'lj' for s_ in srcs); n_em = len(srcs) - n_lj
    if STAGE == 2 and n_lj and n_em:                          # replay: REPLAY_FRAC of samples from LJSpeech, the rest emotional
        w = [REPLAY_FRAC / n_lj if s_ == 'lj' else (1 - REPLAY_FRAC) / n_em for s_ in srcs]
        return DataLoader(ds, batch_size=cfg.batch, sampler=WeightedRandomSampler(w, len(w), replacement=True), collate_fn=collate, num_workers=NW, drop_last=True)
    return DataLoader(ds, batch_size=cfg.batch, shuffle=True, collate_fn=collate, num_workers=NW, drop_last=True)
train_loader = make_train_loader()

model = StyleTTS2Lite(cfg, len(vocab), len(EMOTIONS), len(SPK_LIST)).to(dev); disc = MPD().to(dev)
if STAGE == 2: model.load_state_dict(ck1['model']); disc.load_state_dict(ck1['disc']); print('loaded stage-1 weights (trained', ck1['step'], 'steps)')
print(f'generator {count_params(model):.2f} M params | discriminator {count_params(disc):.2f} M (training only)')
for n in ['text_enc', 'sty_a', 'sty_p', 'aligner', 'dur', 'pros', 'decoder', 'diff']: print(f'  {n:9s} {count_params(getattr(model, n)):.2f} M')

In [ ]:
W = dict(fs=1.0, dur=1.0, f0=1.0, en=1.0, mel=45.0, stft=1.0, emo=STAGE_CFG[STAGE]['w_emo'], diff=1.0, adv=1.0, fm=2.0)
opt_g = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.8, 0.99), weight_decay=1e-2)
opt_d = torch.optim.AdamW(disc.parameters(), lr=cfg.lr, betas=(0.8, 0.99), weight_decay=1e-2)
lam = lambda st: min(1.0, (st + 1) / cfg.warmup) * (0.5 * (1 + math.cos(math.pi * min(1.0, st / cfg.steps))) * 0.9 + 0.1)
sch_g = torch.optim.lr_scheduler.LambdaLR(opt_g, lam); sch_d = torch.optim.lr_scheduler.LambdaLR(opt_d, lam)
step = 0; hist = []

def save_ckpt(name, full=True):
    d = dict(model=model.state_dict(), disc=disc.state_dict(), step=step, vocab=vocab, spk2id=spk2id, cfg=cfg.__dict__)
    if full: d.update(opt_g=opt_g.state_dict(), opt_d=opt_d.state_dict(), sch_g=sch_g.state_dict(), sch_d=sch_d.state_dict())
    torch.save(d, f'{CKPT}/{name}')

if RESUME and os.path.exists(f'{CKPT}/stage{STAGE}_last.pt'):
    ck = torch.load(f'{CKPT}/stage{STAGE}_last.pt', map_location=dev); model.load_state_dict(ck['model']); disc.load_state_dict(ck['disc'])
    opt_g.load_state_dict(ck['opt_g']); opt_d.load_state_dict(ck['opt_d']); sch_g.load_state_dict(ck['sch_g']); sch_d.load_state_dict(ck['sch_d'])
    step = ck['step']; print('resumed stage', STAGE, 'at step', step)

def gan_d_loss(real, fake): return sum(((r - 1) ** 2).mean() + (f ** 2).mean() for r, f in zip(real, fake))
def gan_g_loss(fake): return sum(((f - 1) ** 2).mean() for f in fake)
def fm_loss(fr, ff): return sum(F.l1_loss(a, b.detach()) for r, f in zip(fr, ff) for a, b in zip(f, r))

def train(n_steps):
    global step
    model.train(); disc.train(); t0 = time.time(); acc = {}
    while step < n_steps:
        for b in train_loader:
            if step >= n_steps: break
            b = {k: v.to(dev) for k, v in b.items()}
            L, yg, yr = model.train_forward(b, use_diff=step >= cfg.diff_start)
            gan = step >= cfg.gan_start
            if gan:                                                            # --- discriminator
                dr, _ = disc(yr); df, _ = disc(yg.detach()); ld = gan_d_loss(dr, df)
                opt_d.zero_grad(set_to_none=True); ld.backward(); nn.utils.clip_grad_norm_(disc.parameters(), 5.0); opt_d.step()
            loss = sum(W[k] * v for k, v in L.items())                         # --- generator
            if gan:
                dr, fr = disc(yr); df, ff = disc(yg); L['adv'] = gan_g_loss(df); L['fm'] = fm_loss(fr, ff)
                loss = loss + W['adv'] * L['adv'] + W['fm'] * L['fm']
            opt_g.zero_grad(set_to_none=True); loss.backward()
            gn = nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            if torch.isfinite(gn): opt_g.step()
            sch_g.step(); sch_d.step(); step += 1
            for k, v in L.items(): acc[k] = acc.get(k, 0) + float(v)
            if step % cfg.log_every == 0:
                msg = ' '.join(f'{k}={v / cfg.log_every:.3f}' for k, v in acc.items()); hist.append((step, {k: v / cfg.log_every for k, v in acc.items()}))
                print(f'[{step}/{n_steps}] {msg} | {(time.time() - t0) / cfg.log_every:.2f}s/it'); acc = {}; t0 = time.time()
            if step % cfg.save_every == 0: save_ckpt(f'stage{STAGE}_last.pt')
    save_ckpt(f'stage{STAGE}_last.pt'); save_ckpt(f'stage{STAGE}_final.pt', full=False)      # *_final.pt = weights + vocab only

train(cfg.steps)

In [ ]:
import matplotlib.pyplot as plt
model.eval()
b = next(iter(mk(val_idx, False))); b = {k: v.to(dev) for k, v in b.items()}
with torch.no_grad():
    tm = torch.arange(b['ids'].shape[1], device=dev)[None] < b['tlen'][:, None]; mm = torch.arange(b['mel'].shape[2], device=dev)[None] < b['mlen'][:, None]
    _, emb = model.text_enc(b['ids'], tm); lp, hard = model._align(emb, norm_mel(b['mel']) * mm.unsqueeze(1), tm, b['tlen'], b['mlen'])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); t, m_ = int(b['tlen'][0]), int(b['mlen'][0])
ax[0].imshow(F.softmax(lp[0, :m_, :t], -1).T.cpu(), aspect='auto', origin='lower'); ax[0].set_title('soft alignment (aligner)')
ax[1].imshow(hard[0, :m_, :t].T.cpu(), aspect='auto', origin='lower'); ax[1].set_title('hard alignment (MAS)'); plt.show()
if hist:
    plt.figure(figsize=(6, 3)); [plt.plot([h[0] for h in hist if k in h[1]], [h[1][k] for h in hist if k in h[1]], label=k) for k in ['mel', 'stft', 'fs', 'dur', 'diff', 'emo']]
    plt.legend(); plt.yscale('log'); plt.xlabel('step'); plt.show()

In [ ]:
from IPython.display import Audio, display
def text_to_ids(text): return torch.tensor([[c2i.get(ch, 1) for ch in phonemize([text])[0]]])

@torch.no_grad()
def synthesize(text, emotion='neutral', speaker=None, cfg_scale=1.5, steps=8, ref_wav=None, speed=1.0, m=None):
    m = model if m is None else m; d_ = next(m.parameters()).device
    ids = text_to_ids(text).to(d_)
    spk = torch.tensor([spk2id[speaker] if speaker else 0], device=d_); emo = torch.tensor([E2I[emotion]], device=d_)
    ref = None
    if ref_wav is not None:
        y, _ = librosa.load(ref_wav, sr=cfg.sr); ref = _MEL_CPU(torch.from_numpy(y / (np.abs(y).max() + 1e-8) * 0.95)[None]).to(d_)
    wav, info = m.infer(ids, emo, spk, ref_mel=ref, steps=steps, cfg_scale=cfg_scale, speed=speed)
    return wav.float().cpu().numpy()

def report_unknown(text):
    ph = phonemize([text])[0]; n = sum(c2i.get(ch, 1) == 1 for ch in ph); print(f'{n} unknown symbols of {len(ph)}: {text!r}')

model.eval()
DEMO_TEXTS = ['The weather is really nice today.', 'I can not believe you did that to me.', 'Please leave the package by the front door.']
present_spk = sorted({m['spk_name'] for m in metas}); present_emo = [e for e in EMOTIONS if any(m['emo_name'] == e for m in metas)]
esd_voice = next((s_ for s_ in present_spk if s_.startswith('esd')), 'lj')
for t in DEMO_TEXTS: report_unknown(t)
if STAGE == 1:                                                # text generalisation check (neutral, LJSpeech voice)
    for t in DEMO_TEXTS: print(t); display(Audio(synthesize(t, 'neutral', 'lj'), rate=cfg.sr))
else:                                                         # same NEW sentence, every emotion, in the ESD voice and in the LJSpeech voice
    for v in dict.fromkeys(['lj', esd_voice]):
        for e in present_emo: print(f'{v} | {e}'); display(Audio(synthesize(DEMO_TEXTS[0], e, v), rate=cfg.sr))

In [ ]:
if STAGE == 2:
    emo_ = 'angry' if 'angry' in present_emo else present_emo[-1]
    for w in [1.0, 2.0, 3.0]:
        print(emo_, '| guidance', w); display(Audio(synthesize(DEMO_TEXTS[1], emo_, esd_voice, cfg_scale=w), rate=cfg.sr))

In [ ]:
assert STAGE == 2, 'Stage 1 finished (weights saved as stage1_final.pt). Set STAGE = 2 in the config cell and Run all again.'